# 03 Manual Attention vs PyTorch SDPA

先证明输出近似一致，再理解“相同数学表达式可以有完全不同的工程实现”。

In [1]:
import math
import time
import torch
import torch.nn.functional as F

torch.manual_seed(42)
B, H, T, D = 2, 4, 32, 16
Q = torch.randn(B, H, T, D)
K = torch.randn(B, H, T, D)
V = torch.randn(B, H, T, D)

In [2]:
scores = Q @ K.transpose(-2, -1) / math.sqrt(D)
mask = torch.ones(T, T, dtype=torch.bool).tril()
scores = scores.masked_fill(~mask, float("-inf"))
manual_weights = torch.softmax(scores, dim=-1)
manual_output = manual_weights @ V

sdpa_output = F.scaled_dot_product_attention(
    Q, K, V,
    dropout_p=0.0,
    is_causal=True,
)

print("manual:", manual_output.shape)
print("sdpa:", sdpa_output.shape)
print("max abs error:", (manual_output - sdpa_output).abs().max().item())

manual: torch.Size([2, 4, 32, 16])
sdpa: torch.Size([2, 4, 32, 16])
max abs error: 3.5762786865234375e-07


## 关键区别

`scaled_dot_product_attention` 提供等价的高层语义，但在合适 CUDA 环境可能选择更优化的后端。学习时理解 Manual，工程时优先考虑经过优化的 primitive。